In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = (
    SparkSession
        .builder
        .appName("Fraud Detection AI")
        .getOrCreate()
)

In [3]:
print(spark.version)

4.2.0


In [4]:
spark


In [5]:
df = (
    spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv("../data/raw/creditcard.csv")
)

In [6]:
df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [7]:
df.columns

['Time',
 'V1',
 'V2',
 'V3',
 'V4',
 'V5',
 'V6',
 'V7',
 'V8',
 'V9',
 'V10',
 'V11',
 'V12',
 'V13',
 'V14',
 'V15',
 'V16',
 'V17',
 'V18',
 'V19',
 'V20',
 'V21',
 'V22',
 'V23',
 'V24',
 'V25',
 'V26',
 'V27',
 'V28',
 'Amount',
 'Class']

In [9]:
df.dtypes

[('Time', 'double'),
 ('V1', 'double'),
 ('V2', 'double'),
 ('V3', 'double'),
 ('V4', 'double'),
 ('V5', 'double'),
 ('V6', 'double'),
 ('V7', 'double'),
 ('V8', 'double'),
 ('V9', 'double'),
 ('V10', 'double'),
 ('V11', 'double'),
 ('V12', 'double'),
 ('V13', 'double'),
 ('V14', 'double'),
 ('V15', 'double'),
 ('V16', 'double'),
 ('V17', 'double'),
 ('V18', 'double'),
 ('V19', 'double'),
 ('V20', 'double'),
 ('V21', 'double'),
 ('V22', 'double'),
 ('V23', 'double'),
 ('V24', 'double'),
 ('V25', 'double'),
 ('V26', 'double'),
 ('V27', 'double'),
 ('V28', 'double'),
 ('Amount', 'double'),
 ('Class', 'int')]

In [10]:
df.count()

284807

In [11]:
df.rdd.getNumPartitions()

24

In [12]:
spark.sparkContext.defaultParallelism

24

In [13]:
import os

print(os.cpu_count())

24


In [14]:
df.show(5, truncate=False)

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|V1                |V2                 |V3              |V4                |V5                 |V6                 |V7                 |V8                |V9                |V10                |V11               |V12               |V13               |V14               |V15               |V16               |V17               |V18                |V19               |V20                |V21                 |V22                |V23  

In [15]:
df.groupBy("Class").count().show()

+-----+------+
|Class| count|
+-----+------+
|    1|   492|
|    0|284315|
+-----+------+



In [16]:
total = df.count()

frauds = (
    df.filter(df.Class == 1)
      .count()
)

print(f"Total transactions : {total:,}")
print(f"Fraud transactions : {frauds:,}")
print(f"Fraud percentage   : {frauds / total * 100:.4f}%")

Total transactions : 284,807
Fraud transactions : 492
Fraud percentage   : 0.1727%


In [17]:
print(f"Rows    : {df.count():,}")
print(f"Columns : {len(df.columns)}")

Rows    : 284,807
Columns : 31


In [18]:
from pyspark.sql.functions import col, sum, when

nulls = df.select([
    sum(
        when(col(c).isNull(), 1).otherwise(0)
    ).alias(c)
    for c in df.columns
])

nulls.show(truncate=False)

+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time|V1 |V2 |V3 |V4 |V5 |V6 |V7 |V8 |V9 |V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|0   |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0  |0     |0    |
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [19]:
df.describe().show()

+-------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------+
|summary|             Time|                  V1|                  V2|                  V3|                  V4|                  V5|                  V6|                  V7|                  V8|                  V9|                 V10|                V11|                 V12|                 V13|                 V14|                 V15|     

In [20]:
df.groupBy("Class").count().orderBy("Class").show()

+-----+------+
|Class| count|
+-----+------+
|    0|284315|
|    1|   492|
+-----+------+



In [21]:
df.select("Amount").describe().show()

+-------+-----------------+
|summary|           Amount|
+-------+-----------------+
|  count|           284807|
|   mean|88.34961925093017|
| stddev|250.1201092401885|
|    min|              0.0|
|    max|         25691.16|
+-------+-----------------+



In [22]:
print(f"Partitions: {df.rdd.getNumPartitions()}")

Partitions: 24


In [24]:
df.explain()

== Physical Plan ==
FileScan csv [Time#17,V1#18,V2#19,V3#20,V4#21,V5#22,V6#23,V7#24,V8#25,V9#26,V10#27,V11#28,V12#29,V13#30,V14#31,V15#32,V16#33,V17#34,V18#35,V19#36,V20#37,V21#38,V22#39,V23#40,V24#41,... 6 more fields] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/c:/Users/thoma/Proyectos/fraud-detection-ai/data/raw/creditcard...., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<Time:double,V1:double,V2:double,V3:double,V4:double,V5:double,V6:double,V7:double,V8:doubl...




In [26]:
spark.stop()